# FIFA World Cup Champion Probability Notebook
Goal: predict the probability each remaining country wins the World Cup.
Short flow:
1. Clean match and World Cup history
2. Train a match-win probability model
3. Predict remaining semifinal/final probabilities
4. Simulate 50,000 brackets

In [2]:
%load_ext cudf.pandas
%load_ext cuml.accel

In [21]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

# 1. Data Cleaning

In [38]:
%%cudf.pandas.profile

aliases = {
    "Germany FR": "Germany",
    "German DR": "Germany",
    "rn\">Germany": "Germany",
    "rn\">Republic of Ireland": "Republic of Ireland",
    "Soviet Union": "Russia",
    "Czechoslovakia": "Czech Republic",
    "Serbia and Montenegro": "Serbia",
    "Yugoslavia": "Serbia",
    "IR Iran": "Iran",
    "Korea Republic": "South Korea",
    "Korea DPR": "North Korea",
    "USA": "United States",
    "C�te d'Ivoire": "Cote d'Ivoire",
    "Côte d'Ivoire": "Cote d'Ivoire",
    "Türkiye": "Turkey",
}

matches = pd.read_csv("WorldCupMatches.csv")
worldcups = pd.read_csv("WorldCups.csv")

matches["Year"] = pd.to_numeric(matches["Year"], errors="coerce").astype("Int64")
matches["Home Team Goals"] = pd.to_numeric(matches["Home Team Goals"], errors="coerce")
matches["Away Team Goals"] = pd.to_numeric(matches["Away Team Goals"], errors="coerce")
matches["home_team"] = matches["Home Team Name"].astype(str).str.strip().replace(aliases).str.replace(r'^rn\">', "", regex=True)
matches["away_team"] = matches["Away Team Name"].astype(str).str.strip().replace(aliases).str.replace(r'^rn\">', "", regex=True)
matches["match_date"] = pd.to_datetime(
    matches["Datetime"].astype(str).str.extract(r"(\d{1,2}\s+[A-Za-z]{3}\s+\d{4})")[0],
    format="%d %b %Y",
    errors="coerce",
)

matches = (
    matches
    .drop_duplicates(subset=["MatchID"])
    .dropna(subset=["Year", "home_team", "away_team", "Home Team Goals", "Away Team Goals"])
    .sort_values(["Year", "match_date", "MatchID"])
    .reset_index(drop=True)
)

worldcups["Year"] = pd.to_numeric(worldcups["Year"], errors="coerce").astype("Int64")
worldcups["Winner"] = worldcups["Winner"].fillna("").astype(str).str.strip().replace(aliases)
worldcups.loc[worldcups["Winner"].str.lower().eq("nan").fillna(False), "Winner"] = ""

print(matches.shape, worldcups.shape)
matches.tail()

(1064, 23) (23, 10)


,Year,Datetime,Stage,Stadium,City,Home Team Name,Home Team Goals,Away Team Goals,Away Team Name,Win conditions,...,Referee,Assistant 1,Assistant 2,RoundID,MatchID,Home Team Initials,Away Team Initials,home_team,away_team,match_date
1059,2026,07 Jul 2026 - 13:00,Round of 16,BC Place,Vancouver,Switzerland,0,0,Colombia,Switzerland win on penalties (4 - 3),...,Iván Barton,None,None,202603,2026096,SUI,COL,Switzerland,Colombia,2026-07-07
1060,2026,09 Jul 2026 - 16:00,Quarter-finals,Gillette Stadium,Foxborough,France,2,0,Morocco,None,...,Facundo Tello,None,None,202604,2026097,FRA,MAR,France,Morocco,2026-07-09
1061,2026,10 Jul 2026 - 12:00,Quarter-finals,SoFi Stadium,Inglewood,Spain,2,1,Belgium,None,...,Michael Oliver,None,None,202604,2026098,ESP,BEL,Spain,Belgium,2026-07-10
1062,2026,11 Jul 2026 - 17:00,Quarter-finals,Hard Rock Stadium,Miami Gardens,Norway,1,2,England,England win after extra time,...,Clément Turpin,None,None,202604,2026099,NOR,ENG,Norway,England,2026-07-11
1063,2026,11 Jul 2026 - 20:00,Quarter-finals,Arrowhead Stadium,Kansas City,Argentina,3,1,Switzerland,Argentina win after extra time,...,João Pinheiro,None,None,202604,2026100,ARG,SUI,Argentina,Switzerland,2026-07-11


                                                                                                                  
                                            Total time elapsed: 0.147 seconds                                     
                                          57 GPU function calls in 0.088 seconds                                  
                                          1 CPU function calls in 0.002 seconds                                   
                                                                                                                  
                                                          Stats                                                   
                                                                                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function                     ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ read_csv                     │ 2          │ 0.025       │ 0.012       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.__getitem__        │ 10         │ 0.001       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ to_numeric                   │ 4          │ 0.000       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ NDFrame.astype               │ 6          │ 0.002       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.__setitem__        │ 8          │ 0.003       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ Series                       │ 7          │ 0.000       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ StringMethods.strip          │ 3          │ 0.001       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ NDFrame.replace              │ 3          │ 0.005       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ StringMethods.replace        │ 2          │ 0.001       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ StringMethods.extract        │ 1          │ 0.000       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ to_datetime                  │ 0          │ 0.000       │ 0.000       │ 1          │ 0.002       │ 0.002       │
│ DataFrame.drop_duplicates    │ 1          │ 0.005       │ 0.005       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.dropna             │ 1          │ 0.004       │ 0.004       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.sort_values        │ 1          │ 0.005       │ 0.005       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.reset_index        │ 1          │ 0.000       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ NDFrame.fillna               │ 2          │ 0.001       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ StringMethods.lower          │ 1          │ 0.000       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ Series.eq                    │ 1          │ 0.000       │ 0.000       │ 0          │ 0.000       │ 0.000       │
│ _LocationIndexer.__setitem__ │ 1          │ 0.001       │ 0.001       │ 0          │ 0.000       │ 0.000       │
│ NDFrame.tail                 │ 1          │ 0.003       │ 0.003       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.__repr__           │ 1          │ 0.031       │ 0.031       │ 0          │ 0.000       │ 0.000       │
└──────────────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

Not all pandas operations ran on the GPU. The following functions required CPU fallback:

- to_datetime

To request GPU support for any of these functions, please file a Github issue here: 
]8;id=475156;https://github.com/rapidsai/cudf/issues/new?assignees=&labels=%3F+-+Needs+Triage%2C+feature+request&projects=&template=pandas_function_request.md&title=%5BFEA%5D\https://github.com/rapidsai/cudf/issues/new/choose]8;;\.

# 2. Feature Engineering

In [15]:
features = ["year", "stage_code", "is_knockout", "home_elo", "away_elo", "elo_diff", "title_diff"]

title_history, title_counts = {}, Counter()
for year in sorted(worldcups["Year"].dropna().astype(int).unique()):
    title_history[year] = title_counts.copy()
    winners = worldcups.loc[(worldcups["Year"].eq(year)) & (worldcups["Winner"].ne("")), "Winner"]
    if len(winners):
        title_counts[str(winners.iloc[0]).strip()] += 1

elo = {}
rows = []

for _, m in matches.iterrows():
    year, home, away = int(m["Year"]), str(m["home_team"]), str(m["away_team"])
    elo.setdefault(home, 1500.0)
    elo.setdefault(away, 1500.0)

    stage = str(m["Stage"]).lower()
    stage_code = 4 if "semi" in stage else 3 if "quarter" in stage else 2 if "round of 16" in stage or "second round" in stage else 1 if "round of 32" in stage else 6 if stage.strip() == "final" else 0
    is_knockout = int(stage_code > 0)

    hg, ag = float(m["Home Team Goals"]), float(m["Away Team Goals"])
    if hg > ag:
        y, result = 1.0, 1.0
    elif ag > hg:
        y, result = 0.0, 0.0
    else:
        condition = str(m.get("Win conditions", "")).lower()
        y = 1.0 if home.lower() in condition else 0.0 if away.lower() in condition else np.nan
        result = 0.5

    titles = title_history.get(year, Counter())
    rows.append({
        "Year": year,
        "home_team": home,
        "away_team": away,
        "target_home_wins": y,
        "year": float(year),
        "stage_code": float(stage_code),
        "is_knockout": float(is_knockout),
        "home_elo": elo[home],
        "away_elo": elo[away],
        "elo_diff": elo[home] - elo[away],
        "title_diff": float(titles[home] - titles[away]),
    })

    expected = 1.0 / (1.0 + 10.0 ** (-(elo[home] - elo[away]) / 400.0))
    elo[home] += 28.0 * (result - expected)
    elo[away] += 28.0 * ((1.0 - result) - (1.0 - expected))

feature_table = pd.DataFrame(rows)
feature_table[features] = feature_table[features].fillna(0).astype("float32")
feature_table.tail()

,Year,home_team,away_team,target_home_wins,year,stage_code,is_knockout,home_elo,away_elo,elo_diff,title_diff
1059,2026,Switzerland,Colombia,1.0,2026.0,2.0,1.0,1537.069580,1560.710327,-23.640690,0.0
1060,2026,France,Morocco,1.0,2026.0,3.0,1.0,1721.962769,1533.784424,188.178360,2.0
1061,2026,Spain,Belgium,1.0,2026.0,3.0,1.0,1635.012451,1601.720337,33.292130,1.0
1062,2026,Norway,England,0.0,2026.0,3.0,1.0,1557.564331,1620.628052,-63.063828,-1.0
1063,2026,Argentina,Switzerland,1.0,2026.0,3.0,1.0,1693.819580,1538.020752,155.798843,3.0


# 3a. Train Match Probability Model - Logistic Regression

In [39]:
%%cuml.accel.profile

train = feature_table.dropna(subset=["target_home_wins"])
train = train[train["Year"] <= 2022]

X_train = train[features].astype("float32")
y_train = train["target_home_wins"].astype("int32")

mean = X_train.mean()
std = X_train.std().replace(0, 1)
X_train_scaled = ((X_train - mean) / std).astype("float32")

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_scaled, y_train)

print("training rows:", len(train))

training rows: 780


cuml.accel profile                                                      
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Function               ┃ GPU calls ┃ GPU time ┃ CPU calls ┃ CPU time ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━┩
│ LogisticRegression.fit │         1 │   10.5ms │         0 │       0s │
├────────────────────────┼───────────┼──────────┼───────────┼──────────┤
│ Total                  │         1 │   10.5ms │         0 │       0s │
└────────────────────────┴───────────┴──────────┴───────────┴──────────┘

# 3b. Train Match Probability Model - XGBoost

In [27]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos

baseline_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'device': 'cuda', 
    'max_depth': 6,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'verbosity': 1,
    'scale_pos_weight': scale
}

# Training
dtrain = xgb.DMatrix(X_train_scaled, label=y_train)
# dval = xgb.DMatrix(X_val, label=y_val)

xgb_model = xgb.train(
    baseline_params,
    dtrain,
    num_boost_round=500,
    evals=[(dtrain, 'train')],
    early_stopping_rounds=50,
    verbose_eval=50
)

print("training rows:", len(train))

[0]	train-auc:0.86233
[50]	train-auc:0.95674
[100]	train-auc:0.98554
[150]	train-auc:0.99491
[200]	train-auc:0.99864
[250]	train-auc:0.99948
[300]	train-auc:0.99968
[350]	train-auc:0.99984
[400]	train-auc:0.99994
[450]	train-auc:1.00000
[489]	train-auc:0.99999
training rows: 780


# 4. Predict Remaining Match Probabilities

In [32]:
fixtures = pd.DataFrame([
    ["semifinal", "SF1", "Semi-finals", "France", "Spain"],
    ["semifinal", "SF2", "Semi-finals", "England", "Argentina"],
    ["final", "France|England", "Final", "France", "England"],
    ["final", "France|Argentina", "Final", "France", "Argentina"],
    ["final", "Spain|England", "Final", "Spain", "England"],
    ["final", "Spain|Argentina", "Final", "Spain", "Argentina"],
], columns=["kind", "key", "stage", "home", "away"])

titles_2026 = title_history[2026]
fixtures["year"] = 2026.0
fixtures["stage_code"] = np.where(fixtures["stage"].eq("Final"), 6.0, 4.0)
fixtures["is_knockout"] = 1.0
fixtures["home_elo"] = [elo[t] for t in fixtures["home"]]
fixtures["away_elo"] = [elo[t] for t in fixtures["away"]]
fixtures["elo_diff"] = fixtures["home_elo"] - fixtures["away_elo"]
fixtures["title_diff"] = [titles_2026[h] - titles_2026[a] for h, a in zip(fixtures["home"], fixtures["away"])]

X_future = ((fixtures[features].astype("float32") - mean) / std).astype("float32")

dfuture = xgb.DMatrix(X_future)
fixtures["p_home_advances"] = xgb_model.predict(dfuture)
fixtures["p_away_advances"] = 1 - fixtures["p_home_advances"]

fixtures[["kind", "home", "away", "p_home_advances", "p_away_advances"]]

,kind,home,away,p_home_advances,p_away_advances
0,semifinal,France,Spain,0.514176,0.485824
1,semifinal,England,Argentina,0.090052,0.909948
2,final,France,England,0.447498,0.552502
3,final,France,Argentina,0.075898,0.924102
4,final,Spain,England,0.322470,0.677530
5,final,Spain,Argentina,0.231302,0.768698


# 5. Simulate Champion Probability

In [33]:
rng = np.random.default_rng(42)
n = 50_000

p_sf1 = float(fixtures.loc[fixtures["key"].eq("SF1"), "p_home_advances"].iloc[0])
p_sf2 = float(fixtures.loc[fixtures["key"].eq("SF2"), "p_home_advances"].iloc[0])
final_probs = fixtures[fixtures["kind"].eq("final")].set_index("key")["p_home_advances"].to_dict()

finalist_1 = np.where(rng.random(n) < p_sf1, "France", "Spain")
finalist_2 = np.where(rng.random(n) < p_sf2, "England", "Argentina")
final_keys = [f"{a}|{b}" for a, b in zip(finalist_1, finalist_2)]
final_p = np.array([final_probs[key] for key in final_keys])
champions = np.where(rng.random(n) < final_p, finalist_1, finalist_2)

champion_probabilities = (
    pd.Series(champions, name="team")
    .value_counts(normalize=True)
    .rename_axis("team")
    .reset_index(name="champion_probability")
)

champion_probabilities.to_csv( "champion_probabilities.csv", index=False)
champion_probabilities["champion_probability"] = (champion_probabilities["champion_probability"] * 100).round(2).astype(str) + "%"
champion_probabilities

,team,champion_probability
0,Argentina,77.3%
1,Spain,11.58%
2,England,5.57%
3,France,5.56%
